# exp153_full_rank_slot_addonly_on_exp092 Colab Train

Colab-first runner for full rank-slot add-only training on the exp092 LightGBM surface.

## 1. Mount Drive and check runtime

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

from pathlib import Path
import psutil

try:
    import torch
except Exception:
    torch = None

DRIVE_ROOT = Path("/content/drive/MyDrive/Kaggle/ROGII")
EXP_NAME = "exp153_full_rank_slot_addonly_on_exp092"
EXP_DIR = DRIVE_ROOT / "experiments" / EXP_NAME
CACHE_SOURCE = DRIVE_ROOT / "experiments/exp072_exp063_full_replay_feature_cache/artifacts/exp063_full_replay_feature_cache_pixiux_likpf_public_replay_train_features.csv.gz"
LOCAL_CACHE_DIR = Path("/content/rogii_cache/exp072_artifacts")
LOCAL_CACHE = LOCAL_CACHE_DIR / CACHE_SOURCE.name

print("drive_root:", DRIVE_ROOT, DRIVE_ROOT.exists())
print("experiment_dir:", EXP_DIR, EXP_DIR.exists())
print("RAM GB:", round(psutil.virtual_memory().total / 1024**3, 2))
if torch is not None:
    print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)


## 2. Install dependencies

In [ ]:
!pip install -q numpy pandas pyyaml scikit-learn matplotlib lightgbm psutil


## 3. Validate Drive layout

In [ ]:
required = [
    DRIVE_ROOT / "project.yml",
    DRIVE_ROOT / "data/raw/train",
    EXP_DIR / "config.yaml",
    EXP_DIR / "settings.py",
    EXP_DIR / "full_rank_slot_addonly_on_exp092.py",
    CACHE_SOURCE,
]

for path in required:
    print(path, path.exists(), path.stat().st_size if path.exists() and path.is_file() else "")

missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required Colab inputs:\n" + "\n".join(missing))

print("train_files:", len(list((DRIVE_ROOT / "data/raw/train").glob("*"))))


## 4. Copy large cache to /content

In [ ]:
import shutil
import time
import pandas as pd

LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
if not LOCAL_CACHE.exists() or LOCAL_CACHE.stat().st_size != CACHE_SOURCE.stat().st_size:
    t0 = time.time()
    shutil.copy2(CACHE_SOURCE, LOCAL_CACHE)
    print("copied cache seconds:", round(time.time() - t0, 2))
else:
    print("local cache already present")

print("local_cache:", LOCAL_CACHE, LOCAL_CACHE.exists(), LOCAL_CACHE.stat().st_size)
preview = pd.read_csv(LOCAL_CACHE, nrows=3, dtype={"id": str, "well": str})
display(preview[[c for c in ["id", "well", "target", "last_known_tvt", "pf_ancc", "beam_mean_d", "likpf_mean_d"] if c in preview.columns]])


## 5. LightGBM GPU smoke test

In [ ]:
import numpy as np
from lightgbm import LGBMRegressor

rng = np.random.default_rng(42)
X = rng.normal(size=(5000, 16)).astype(np.float32)
y = (X[:, 0] * 0.7 - X[:, 1] * 0.2 + rng.normal(scale=0.1, size=5000)).astype(np.float32)
model = LGBMRegressor(objective="regression", n_estimators=20, num_leaves=31, learning_rate=0.1, device_type="gpu", gpu_use_dp=True, verbose=-1)
model.fit(X, y)
print("lightgbm_gpu_smoke_ok", model.booster_.current_iteration())


## 6. Start background full train

In [ ]:
import json
import subprocess
import textwrap
import time

RUN_DIR = EXP_DIR / "colab_runs"
RUN_DIR.mkdir(parents=True, exist_ok=True)

run_id = time.strftime("run_%Y%m%d_%H%M%S_highmem_local_cache")
run_py = RUN_DIR / f"{run_id}_{EXP_NAME}_full_train.py"
log_path = RUN_DIR / f"{run_id}_{EXP_NAME}_full_train.log"
pid_path = RUN_DIR / f"{run_id}_pid.txt"
latest_path = RUN_DIR / "latest_run.json"

run_py.write_text(textwrap.dedent(f'''
    from pathlib import Path
    import json
    import os
    import sys
    import time
    import traceback

    ROOT = Path("{DRIVE_ROOT}")
    EXP_NAME = "{EXP_NAME}"
    EXP = ROOT / "experiments" / EXP_NAME
    LOCAL_CACHE = Path("{LOCAL_CACHE}")
    os.chdir(ROOT)
    sys.path.insert(0, str(EXP))

    from settings import ExperimentPaths, get_nested, load_config
    from full_rank_slot_addonly_on_exp092 import run_full_rank_slot_addonly_on_exp092

    def cfg_get(config, dotted_key, default=None):
        value = get_nested(config, dotted_key)
        return default if value is None else value

    try:
        paths = ExperimentPaths()
        paths.ensure_output_dirs()
        config = load_config()
        print("START full rank-slot add-only Colab train", flush=True)
        print("cwd=", Path.cwd(), flush=True)
        print("local_cache_exists=", LOCAL_CACHE.exists(), "size=", LOCAL_CACHE.stat().st_size if LOCAL_CACHE.exists() else None, flush=True)
        print("active_modes=", cfg_get(config, "model.training.active_modes"), flush=True)
        print("active_variants=", [v["name"] for v in cfg_get(config, "model.feature_ablation.active_variants", [])], flush=True)

        t0 = time.time()
        summary = run_full_rank_slot_addonly_on_exp092(
            output_dir=paths.artifacts_dir,
            train_dir=paths.train_data_dir,
            cache_path=LOCAL_CACHE,
            projection_config=cfg_get(config, "model.u_projection", {{}}),
            rank_slot_config=cfg_get(config, "model.rank_slot", {{}}),
            variants=cfg_get(config, "model.feature_ablation.active_variants", []),
            modes=cfg_get(config, "model.training.modes", {{}}),
            active_modes=cfg_get(config, "model.training.active_modes", []),
            n_splits=int(cfg_get(config, "validation.n_folds", 5)),
            fast=bool(cfg_get(config, "audit.fast", False)),
            early_stopping_rounds=int(cfg_get(config, "model.training.early_stopping_rounds", 250)),
            max_rows=cfg_get(config, "model.training.max_rows"),
            max_train_rows=cfg_get(config, "model.training.max_train_rows"),
            save_models=bool(cfg_get(config, "model.training.save_models", True)),
            save_predictions=bool(cfg_get(config, "model.training.save_predictions", True)),
            top_n_importance=int(cfg_get(config, "model.training.top_n_importance", 50)),
        )
        summary["colab_elapsed_seconds_outer"] = round(time.time() - t0, 3)
        paths.metrics_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False, sort_keys=True) + "\\n")
        (EXP / "colab_runs/latest_done_summary.json").write_text(json.dumps(summary, indent=2))
        print("DONE full rank-slot add-only Colab train", flush=True)
        print(json.dumps(summary, indent=2)[:12000], flush=True)
    except Exception:
        (EXP / "colab_runs/latest_failed.txt").write_text(traceback.format_exc())
        print("FAILED full rank-slot add-only Colab train", flush=True)
        traceback.print_exc()
        raise
'''))

cmd = f"python {run_py} > {log_path} 2>&1"
proc = subprocess.Popen(["bash", "-lc", cmd], cwd=str(DRIVE_ROOT), start_new_session=True)
pid_path.write_text(str(proc.pid))

latest = {
    "run_id": run_id,
    "pid": proc.pid,
    "run_py": str(run_py),
    "log_path": str(log_path),
    "pid_path": str(pid_path),
    "local_cache": str(LOCAL_CACHE),
    "started_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}
latest_path.write_text(json.dumps(latest, indent=2))
print(json.dumps(latest, indent=2))


## 7. Check status and logs

In [ ]:
import json
import subprocess

RUN_DIR = EXP_DIR / "colab_runs"
latest = json.loads((RUN_DIR / "latest_run.json").read_text())
print(json.dumps(latest, indent=2))

pid = str(latest["pid"])
print(subprocess.run(["ps", "-p", pid, "-o", "pid,ppid,stat,etime,time,%cpu,%mem,rss,cmd"], capture_output=True, text=True).stdout)

log = Path(latest["log_path"])
print("log:", log, log.exists(), log.stat().st_size if log.exists() else None)
if log.exists():
    print("\n".join(log.read_text(errors="replace").splitlines()[-120:]))

failed = RUN_DIR / "latest_failed.txt"
done = RUN_DIR / "latest_done_summary.json"
print("failed:", failed.exists(), failed.stat().st_size if failed.exists() else None)
print("done:", done.exists(), done.stat().st_size if done.exists() else None)
